In [1]:
# src/train_lstm.py
import sys
import numpy as np
import joblib
from pathlib import Path
from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# root dir
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loader import load_and_process_data

# config for model training
LOOKBACK = 60
FUTURE_DAYS = 5
TARGET_THRESHOLD = 0.015
TRAIN_SPLIT = 0.8

SRC_DIR = PROJECT_ROOT / "src"
MODEL_NAME = SRC_DIR / "ShortTerm_LSTM_5d_v1.keras"
ARTIFACTS_NAME = SRC_DIR / "scaler_and_features.joblib"

print("Loading and Processing Data...")
X, y, dates, feat_cols, returns = load_and_process_data(
    lookback=LOOKBACK,
    future_days=FUTURE_DAYS,
    target_thresh=TARGET_THRESHOLD
)

print(f"Data Loaded. Sequences: {X.shape}, Targets: {y.shape}")

# split based on time
unique_dates = np.sort(np.unique(dates))
cutoff_idx = int(len(unique_dates) * TRAIN_SPLIT)
cutoff_date = unique_dates[cutoff_idx]
print(f"Train/Test Split Date: {cutoff_date}")

train_mask = dates <= cutoff_date
test_mask = dates > cutoff_date

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
# scaling
scaler = StandardScaler()
n_train, lookback, n_features = X_train.shape
n_test = X_test.shape[0]

X_train_2d = X_train.reshape(-1, n_features)
X_test_2d = X_test.reshape(-1, n_features)
scaler.fit(X_train_2d) 

# tranform both
X_train_scaled = scaler.transform(X_train_2d)
X_test_scaled = scaler.transform(X_test_2d)

# reshape back to 3D
X_train = X_train_scaled.reshape(n_train, lookback, n_features)
X_test = X_test_scaled.reshape(n_test, lookback, n_features)
returns_test = returns[test_mask]

# class weights (to handle imbalance)
cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights_dict = {0: cw[0], 1: cw[1]}

# building the model
print("Building LSTM...")
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    LSTM(32, return_sequences=False),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Training...")
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=64,
    class_weight=weights_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("Evaluation...")
y_pred_prob = model.predict(X_test, verbose=0).reshape(-1)

thresholds = [0.4, 0.45, 0.5, 0.55, 0.6]
for thresh in thresholds:
    y_pred = (y_pred_prob > thresh).astype(int)

    print(f"\nThreshold: {thresh:.2f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    buy_mask = y_pred == 1
    avg_return_buy = returns_test[buy_mask].mean() if buy_mask.any() else 0.0
    avg_return_mkt = returns_test.mean()
    print(f"Market Avg Return (5d): {avg_return_mkt:.4f}")
    print(f"Model Buy Return (5d):  {avg_return_buy:.4f}")

print("Saving Artifacts...")
model.save(str(MODEL_NAME))
joblib.dump({'scaler': scaler, 'feature_columns': feat_cols}, str(ARTIFACTS_NAME))
print(f"Model saved to: {MODEL_NAME}")
print(f"Artifacts saved to: {ARTIFACTS_NAME}")
print("Done.")


Loading and Processing Data...
Data Loaded. Sequences: (10880, 60, 18), Targets: (10880,)
Train/Test Split Date: 2024-02-15 00:00:00
Building LSTM...


c:\Users\akw97\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training...
Epoch 1/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 18s 58ms/step - accuracy: 0.5395 - loss: 0.6903 - val_accuracy: 0.5032 - val_loss: 0.7037 - learning_rate: 0.0010
Epoch 2/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - accuracy: 0.5556 - loss: 0.6850 - val_accuracy: 0.5898 - val_loss: 0.6738 - learning_rate: 0.0010
Epoch 3/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 7s 48ms/step - accuracy: 0.5628 - loss: 0.6826 - val_accuracy: 0.5345 - val_loss: 0.6918 - learning_rate: 0.0010
Epoch 4/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 11s 78ms/step - accuracy: 0.5699 - loss: 0.6796 - val_accuracy: 0.5608 - val_loss: 0.6879 - learning_rate: 0.0010
Epoch 5/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.5612 - loss: 0.6788 - val_accuracy: 0.5797 - val_loss: 0.6798 - learning_rate: 0.0010
Epoch 6/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.5674 - loss: 0.6793 - val_accuracy: 0.5866 - val_loss: 0.6852 - learning_rate: 0.0010
Epoch 7/50
137/137 ━━━━━━━━━━━━━━━━━━━━ 14s 103ms/step - accuracy: 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("Flattening 3D sequence data for traditional models...")
# (sample, 60, 18) into (sample, 1080), 3d to 2d
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42)
rf_model.fit(X_train_flat, y_train)

print("Training Logistic Regression...")
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_flat, y_train)

print("Generating Probabilities...\n")
rf_probs = rf_model.predict_proba(X_test_flat)[:, 1]
lr_probs = lr_model.predict_proba(X_test_flat)[:, 1]

# looping through models
models = {
    'LSTM': y_pred_prob,  #lstm probs from last cell
    'Random Forest': rf_probs,
    'Logistic Regression': lr_probs
}

# thresholds to test
THRESHOLDS = [0.4, 0.45, 0.5, 0.55]

for threshold in THRESHOLDS:

    print(f"========== MODEL COMPARISON (Threshold: {threshold}) ==========\n")

    for model_name, probs in models.items():
        print(f"--- {model_name} ---")
    
        y_pred = (probs > threshold).astype(int)
    
        print("Classification Report:")
        print(classification_report(y_test, y_pred))
    
        buy_mask = y_pred == 1
        avg_return_buy = returns_test[buy_mask].mean() if buy_mask.any() else 0.0
        avg_return_mkt = returns_test.mean()
    
        print(f"Market Avg Return (5d): {avg_return_mkt:.4f}")
        print(f"Model Buy Return (5d):  {avg_return_buy:.4f}")
        print("\n" + "="*55 + "\n")

Flattening 3D sequence data for traditional models...
Training Random Forest...
Training Logistic Regression...
Generating Probabilities...

========== MODEL COMPARISON (Threshold: 0.4) ==========

--- LSTM ---
Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.43      0.52      1299
           1       0.43      0.63      0.51       873

    accuracy                           0.51      2172
   macro avg       0.53      0.53      0.51      2172
weighted avg       0.55      0.51      0.51      2172

Market Avg Return (5d): 0.0067
Model Buy Return (5d):  0.0082


--- Random Forest ---
Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.08      0.14      1299
           1       0.40      0.91      0.55       873

    accuracy                           0.41      2172
   macro avg       0.48      0.49      0.35      2172
weighted avg       0.50      0.41      0.31      2172

Mar

In [7]:
# Showing the evaluation results in a table
results = []

for threshold in THRESHOLDS:
    for model_name, probs in models.items():
        y_pred = (probs > threshold).astype(int)
        buy_mask = y_pred == 1
        avg_return_buy = returns_test[buy_mask].mean() if buy_mask.any() else 0.0
        avg_return_mkt = returns_test.mean()
        results.append({
            'Threshold': threshold,
            'Model': model_name,
            'Market Return': avg_return_mkt,
            'Model BUY Return': avg_return_buy
        })

df_results = pd.DataFrame(results)
print(df_results)
# table format 
pivot_df = df_results.pivot(index='Threshold', columns='Model', values=['Market Return', 'Model BUY Return'])
display(pivot_df.round(4))

    Threshold                Model  Market Return  Model BUY Return
0        0.40                 LSTM       0.006709          0.008216
1        0.40        Random Forest       0.006709          0.006624
2        0.40  Logistic Regression       0.006709          0.006330
3        0.45                 LSTM       0.006709          0.011939
4        0.45        Random Forest       0.006709          0.009208
5        0.45  Logistic Regression       0.006709          0.006643
6        0.50                 LSTM       0.006709          0.013503
7        0.50        Random Forest       0.006709          0.014518
8        0.50  Logistic Regression       0.006709          0.007543
9        0.55                 LSTM       0.006709          0.020587
10       0.55        Random Forest       0.006709          0.005365
11       0.55  Logistic Regression       0.006709          0.004276


Market Return                                   Model BUY Return  \
Model              LSTM Logistic Regression Random Forest             LSTM   
Threshold                                                                    
0.40             0.0067              0.0067        0.0067           0.0082   
0.45             0.0067              0.0067        0.0067           0.0119   
0.50             0.0067              0.0067        0.0067           0.0135   
0.55             0.0067              0.0067        0.0067           0.0206   

                                             
Model     Logistic Regression Random Forest  
Threshold                                    
0.40                   0.0063        0.0066  
0.45                   0.0066        0.0092  
0.50                   0.0075        0.0145  
0.55                   0.0043        0.0054